# Milestone 2: Cybercrime Complaint Classification & Semantic Retrieval Benchmarking

**Master's Thesis Research**  
*Automated Cybercrime Complaint Classification and Semantic Retrieval: A Comparative Study of Traditional NLP and Transformer-Based Models*  

This notebook provides the end-to-end experimental walkthrough for:
1. **Exploratory Verification** of the 7-class harmonized cybercrime corpus.
2. **Classical ML Baselines**: TF-IDF + MultinomialNB, Logistic Regression, LinearSVC, Random Forest.
3. **Transformer Fine-Tuning**: DistilBERT on Apple Silicon MPS hardware.
4. **Information Retrieval Benchmark**: Sparse BM25 vs Dense Bi-Encoder (all-MiniLM-L6-v2) vs Hybrid RRF.
5. **Interactive Inference & Case Retrieval**: Live victim narrative triage.

In [ ]:
import sys
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project root is accessible
PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

print("Environment initialized successfully.")

## 1. Load Dataset Splits & Distribution Overview

In [ ]:
train_df = pd.read_parquet(PROJECT_ROOT / "data/processed/splits_7class/train.parquet")
val_df = pd.read_parquet(PROJECT_ROOT / "data/processed/splits_7class/val.parquet")
test_df = pd.read_parquet(PROJECT_ROOT / "data/processed/splits_7class/test.parquet")

print(f"Train samples: {len(train_df):,}")
print(f"Val samples  : {len(val_df):,}")
print(f"Test samples : {len(test_df):,}")

fig, ax = plt.subplots(figsize=(10, 4))
sns.countplot(data=train_df, y="primary_category", order=train_df["primary_category"].value_counts().index, palette="viridis", ax=ax)
ax.set_title("Training Split Class Distribution (Stratified)")
ax.set_xlabel("Count")
ax.set_ylabel("Cybercrime Category")
plt.tight_layout()
plt.show()

## 2. Classical Machine Learning Benchmark Results
We load the results generated across the 4 traditional classifiers on the held-out test split ($n=711$).

In [ ]:
ml_results = pd.read_csv(PROJECT_ROOT / "reports/classification_baseline_results.csv")
display(ml_results)

## 3. Information Retrieval (IR) Evaluation Benchmark
Comparison of Sparse (BM25, TF-IDF), Dense Bi-Encoder (`all-MiniLM-L6-v2`), and Hybrid RRF against 10,500 relevance judgments.

In [ ]:
ir_results = pd.read_csv(PROJECT_ROOT / "reports/retrieval_comprehensive_results.csv")
display(ir_results)

# Plot NDCG@10 and MRR@10 comparison
fig, ax = plt.subplots(figsize=(9, 4))
ir_melted = pd.melt(ir_results, id_vars=["Retriever"], value_vars=["NDCG@10", "MRR@10"], var_name="Metric", value_name="Score")
sns.barplot(data=ir_melted, x="Retriever", y="Score", hue="Metric", palette="Set2", ax=ax)
ax.set_title("Information Retrieval Performance (BEIR Benchmark)")
ax.set_ylabel("Evaluation Score")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()

## 4. Live Cybercrime Triage & Case Retrieval Demo
Input an arbitrary victim narrative to see automated PII masking, category classification, and similar case retrieval.

In [ ]:
from scripts.predict_and_retrieve import triage_complaint

narrative = (
    "I got an urgent WhatsApp message claiming to be JPMorgan Chase alerting me to an unauthorized "
    "wire transfer of $12,500. They told me to click http://chase-security-verify.net/login to confirm my SSN."
)
triage_complaint(narrative, top_k=3)